# Data Cleaning

In this stage, we prepare the dataset for further analysis by:

- Removing irrelevant and leakage features
- Handling missing values
- Fixing data consistency issues
- Preserving the original feature representation

Note:
Feature encoding and feature creation will be performed later in the Feature Engineering stage.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# Display formatting
from IPython.core.display import HTML
HTML("""
<style>
.dataframe table, .dataframe th, .dataframe td { font-size: 12px; }
div.output_scroll { overflow-x: auto; }
</style>
""")

In [2]:
df = pd.read_csv("telco.csv")
df.head()

,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,City,Zip Code,Latitude,Longitude,Population,Quarter,Referred a Friend,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,Los Angeles,90022,34.023810,-118.156582,68701,Q3,No,0,1,NaN,No,0.00,No,Yes,DSL,8,No,No,Yes,No,No,Yes,No,No,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,Los Angeles,90063,34.044271,-118.185237,55668,Q3,Yes,1,8,Offer E,Yes,48.85,Yes,Yes,Fiber Optic,17,No,Yes,No,No,No,No,No,Yes,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,Los Angeles,90065,34.108833,-118.229715,47534,Q3,No,0,18,Offer D,Yes,11.33,Yes,Yes,Fiber Optic,52,No,No,No,No,Yes,Yes,Yes,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,Inglewood,90303,33.936291,-118.332639,27778,Q3,Yes,1,25,Offer C,Yes,19.76,No,Yes,Fiber Optic,12,No,Yes,Yes,No,Yes,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,Whittier,90602,33.972119,-118.020188,26265,Q3,Yes,1,37,Offer C,Yes,6.33,Yes,Yes,Fiber Optic,14,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [3]:
data = df.copy()

In [4]:
data.columns = (
    data.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
)

In [5]:
data.columns

Index(['customer_id', 'gender', 'age', 'under_30', 'senior_citizen', 'married',
       'dependents', 'number_of_dependents', 'country', 'state', 'city',
       'zip_code', 'latitude', 'longitude', 'population', 'quarter',
       'referred_a_friend', 'number_of_referrals', 'tenure_in_months', 'offer',
       'phone_service', 'avg_monthly_long_distance_charges', 'multiple_lines',
       'internet_service', 'internet_type', 'avg_monthly_gb_download',
       'online_security', 'online_backup', 'device_protection_plan',
       'premium_tech_support', 'streaming_tv', 'streaming_movies',
       'streaming_music', 'unlimited_data', 'contract', 'paperless_billing',
       'payment_method', 'monthly_charge', 'total_charges', 'total_refunds',
       'total_extra_data_charges', 'total_long_distance_charges',
       'total_revenue', 'satisfaction_score', 'customer_status', 'churn_label',
       'churn_score', 'cltv', 'churn_category', 'churn_reason'],
      dtype='object')

### Column Standardization

Column names were converted to lowercase and spaces were replaced with underscores to improve readability and ensure consistency across the project.

In [6]:
columns_to_drop = [
    "customer_id",
    "country",
    "state",
    "quarter",
    "churn_score",
    "churn_category",
    "churn_reason",
    "customer_status",
    "cltv"
]

data = data.drop(columns=columns_to_drop)

### Feature Removal Strategy

The following columns were removed because they are:

- Unique identifiers
- Constant-value features
- Target leakage features
- Information generated after churn occurred
  
**Note:**

The timing of Satisfaction Score collection is not explicitly documented.

To evaluate its impact and potential leakage risk, two models will be developed later:

- Model A: Excluding Satisfaction Score
- Model B: Including Satisfaction Score

In [7]:
missing_percent = (
    data.isna()
        .sum()
        .sort_values(ascending=False)
        / len(data) * 100
)

missing_percent[missing_percent > 0]

offer            55.047565
internet_type    21.666903
dtype: float64

In [8]:
data["offer"] = data["offer"].fillna("No Offer")

data["internet_type"] = data["internet_type"].fillna("No Internet")

### Missing Values Strategy

- Offer → No Offer

  Missing values indicate that the customer did not receive any marketing offer.

- Internet Type → No Internet

  Missing values indicate that the customer does not subscribe to internet services.

In [9]:
print("Shape:", data.shape)
print("Missing Values:", data.isna().sum().sum())

Shape: (7043, 41)
Missing Values: 0


In [10]:
data.to_csv("telco_cleaned.csv", index=False)

## Final Dataset Summary

- Rows: 7043
- Columns: 41
- Missing values: 0
- Column names standardized
- Irrelevant and leakage features removed
- Original categorical features preserved

The dataset is now clean and ready for exploratory data analysis and feature engineering.